# Categorical Encoding: One-Hot Encoding (OHE)

**One-Hot Encoding** transforms categorical variables into a set of binary indicator features ($0$ or $1$). For a categorical feature with $k$ unique categories, it creates new binary columns indicating the presence of each category.

---

## Learning Objectives
By the end of this notebook, you will be able to:
1. Understand when to use One-Hot Encoding (nominal features with low-to-moderate cardinality).
2. Recognize the **dummy variable trap** (multicollinearity) and know when to drop the first category (`drop='first'`).
3. Understand why `pd.get_dummies()` is risky in machine learning workflows compared to scikit-learn's `OneHotEncoder`.
4. Handle unseen categories in production using `handle_unknown='ignore'`.
5. Build a robust preprocessing workflow using **`ColumnTransformer`** to encode categorical features while passing numerical features through intact.

---
## 1. Concepts, Trade-offs & The Dummy Variable Trap

### When to Use:
- **Nominal categorical variables**: Variables with **no intrinsic order** (e.g., `Embarked` $\in$ {`S`, `C`, `Q`}, `Gender` $\in$ {`male`, `female`}).
- **Low-to-moderate cardinality**: When the number of unique categories is small ($< 15-20$).

### Key Considerations:
1. **High Cardinality Penalty**: If a variable has 1,000 unique values (e.g., zip codes), One-Hot Encoding creates 1,000 sparse columns, dramatically inflating memory usage and slowing down training (the curse of dimensionality).
2. **The Dummy Variable Trap ($k$ vs. $k-1$ dummy variables)**:
   - If a feature has 2 categories (`male`, `female`), knowing that `male = 0` automatically implies `female = 1`.
   - Including both columns introduces **perfect multicollinearity** ($x_{\text{male}} + x_{\text{female}} = 1$), which causes matrix inversion issues in unregularized linear and logistic regression models.
   - For **linear models**, use `drop='first'`.
   - For **tree-based models** and **regularized models ($L_1/L_2$)**, keeping all categories (`drop=None`) is standard practice and often yields better splits.

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

pd.set_option('display.max_columns', None)

---
## 2. Loading the Dataset & Splitting Train/Test Sets

We use the Titanic survival dataset. 
- Categorical features: `Gender`, `Embarked`, `CabinDeck`.
- Numerical features: `Age`, `Fare`, `SibSp`, `Parch`, `Pclass`.

In [2]:
# Robust local path with online fallback
data_path = '../datasets/titanic.csv'
if not os.path.exists(data_path):
    data_path = 'https://raw.githubusercontent.com/cmps460s26/cmps460-content/refs/heads/main/examples/02.eda/datasets/titanic.csv'

# Load Titanic data
usecols = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked", "Cabin", "Survived"]
# Rename 'Sex' to 'Gender' if present for clarity
data = pd.read_csv(data_path)
if 'Sex' in data.columns and 'Gender' not in data.columns:
    data.rename(columns={'Sex': 'Gender'}, inplace=True)

# Select relevant columns
cols_to_keep = [c for c in ["Pclass", "Gender", "Age", "SibSp", "Parch", "Fare", "Embarked", "Cabin", "Survived"] if c in data.columns]
data = data[cols_to_keep].copy()

# Extract the deck letter from Cabin (e.g. 'C85' -> 'C')
data["CabinDeck"] = data["Cabin"].astype(str).str[0]
data.loc[data["CabinDeck"] == 'n', "CabinDeck"] = "Missing"
data.drop("Cabin", axis=1, inplace=True)

# Impute simple baseline for Age and Embarked to focus on encoding
data["Embarked"] = data["Embarked"].fillna("Missing")
data["Age"] = data["Age"].fillna(data["Age"].median())

print(f"Dataset shape: {data.shape}")
data.head()

Dataset shape: (891, 9)


,Pclass,Gender,Age,SibSp,Parch,Fare,Embarked,Survived,CabinDeck
0,3,male,22.0,1,0,7.2500,S,0,NaN
1,1,female,38.0,1,0,71.2833,C,1,C
2,3,female,26.0,0,0,7.9250,S,1,NaN
3,1,female,35.0,1,0,53.1000,S,1,C
4,3,male,35.0,0,0,8.0500,S,0,NaN


In [3]:
# Separate predictors and target
X = data.drop("Survived", axis=1)
y = data["Survived"]

# Split into 70% train and 30% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

X_train shape: (623, 8)
X_test shape:  (268, 8)


---
## 3. Why `pd.get_dummies()` is Risky in Machine Learning

Beginners often use `pd.get_dummies(df)`. While convenient for quick exploratory data analysis, `pd.get_dummies()` has a fatal flaw in production:

> If the test set has **fewer or different categories** than the training set, `pd.get_dummies()` produces **different columns** in `X_train` and `X_test`! The model will crash during prediction because feature counts and column names do not match.

In [4]:
# Illustrating the pd.get_dummies() mismatch problem:
toy_train = pd.DataFrame({'City': ['Doha', 'Dubai', 'Riyadh']})
toy_test  = pd.DataFrame({'City': ['Doha', 'Dubai']})  # Riyadh never appears in test!

print("pd.get_dummies(toy_train) columns:")
print(pd.get_dummies(toy_train).columns.tolist())

print("\npd.get_dummies(toy_test) columns:")
print(pd.get_dummies(toy_test).columns.tolist())
print("-> Mismatch! Test set has 2 columns, but the trained model expects 3 columns!")

pd.get_dummies(toy_train) columns:
['City_Doha', 'City_Dubai', 'City_Riyadh']

pd.get_dummies(toy_test) columns:
['City_Doha', 'City_Dubai']
-> Mismatch! Test set has 2 columns, but the trained model expects 3 columns!


---
## 4. Scikit-Learn `OneHotEncoder`: Safe, Robust Encoding

Scikit-learn's `OneHotEncoder` solves this issue:
1. `fit()` learns the exact categories from `X_train`.
2. `transform()` ensures that `X_test` receives the **exact same feature columns** in the exact same order.
3. `handle_unknown='ignore'`: If an unseen category appears in test data, it sets all dummy columns to $0$ instead of crashing!

In [5]:
# Identify categorical vs numerical columns
cat_cols = ['Gender', 'Embarked', 'CabinDeck']
num_cols = [c for c in X_train.columns if c not in cat_cols]

print(f"Categorical features: {cat_cols}")
print(f"Numerical features:   {num_cols}")

Categorical features: ['Gender', 'Embarked', 'CabinDeck']
Numerical features:   ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']


In [6]:
# Initialize OneHotEncoder with handle_unknown='ignore' and pandas output
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe.set_output(transform="pandas")

# Fit on training categorical columns
ohe.fit(X_train[cat_cols])

# Transform categorical columns
X_train_ohe = ohe.transform(X_train[cat_cols])
X_test_ohe  = ohe.transform(X_test[cat_cols])

print(f"Original categorical column count: {len(cat_cols)}")
print(f"Encoded binary column count:        {X_train_ohe.shape[1]}")
X_train_ohe.head()

Original categorical column count: 3
Encoded binary column count:        15


,Gender_female,Gender_male,Embarked_C,Embarked_Missing,Embarked_Q,Embarked_S,CabinDeck_A,CabinDeck_B,CabinDeck_C,CabinDeck_D,CabinDeck_E,CabinDeck_F,CabinDeck_G,CabinDeck_T,CabinDeck_nan
857,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
52,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
386,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
124,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
578,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [7]:
# Inspect learned categories per feature
for col, cats in zip(cat_cols, ohe.categories_):
    print(f"- {col:12s} categories: {list(cats)}")

- Gender       categories: ['female', 'male']
- Embarked     categories: ['C', 'Missing', 'Q', 'S']
- CabinDeck    categories: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'T', nan]


---
## 5. Production Workflow: `ColumnTransformer`

In real-world applications, we do not encode categorical columns in isolation. We use **`ColumnTransformer`** to:
- Apply `OneHotEncoder` to categorical features.
- Pass numerical features through unchanged (`remainder='passthrough'`).

In [8]:
# Define ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_cols)
    ],
    remainder='passthrough'  # Keeps numerical columns intact
)
preprocessor.set_output(transform="pandas")

# Fit on X_train, transform both sets
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed  = preprocessor.transform(X_test)

print(f"Final preprocessed feature shape: {X_train_preprocessed.shape}")
X_train_preprocessed.head()

Final preprocessed feature shape: (623, 20)


,cat__Gender_female,cat__Gender_male,cat__Embarked_C,cat__Embarked_Missing,cat__Embarked_Q,cat__Embarked_S,cat__CabinDeck_A,cat__CabinDeck_B,cat__CabinDeck_C,cat__CabinDeck_D,cat__CabinDeck_E,cat__CabinDeck_F,cat__CabinDeck_G,cat__CabinDeck_T,cat__CabinDeck_nan,remainder__Pclass,remainder__Age,remainder__SibSp,remainder__Parch,remainder__Fare
857,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1,51.0,0,0,26.5500
52,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1,49.0,1,0,76.7292
386,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3,1.0,5,2,46.9000
124,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1,54.0,0,1,77.2875
578,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3,28.0,1,0,14.4583


---
## 6. Avoiding Multicollinearity: `drop='first'` for Linear Models

When preparing data for linear regression or unregularized logistic regression, we set `drop='first'` to avoid the dummy variable trap.

In [9]:
# One-hot encoding with drop='first'
ohe_drop = OneHotEncoder(sparse_output=False, drop='first')
ohe_drop.set_output(transform="pandas")

ohe_drop.fit(X_train[['Gender', 'Embarked']])
X_train_dropped = ohe_drop.transform(X_train[['Gender', 'Embarked']])

print("Columns generated with drop='first':")
print(X_train_dropped.columns.tolist())
print("Notice 'Gender_male' is kept while 'Gender_female' is dropped as reference category.")
X_train_dropped.head()

Columns generated with drop='first':
['Gender_male', 'Embarked_Missing', 'Embarked_Q', 'Embarked_S']
Notice 'Gender_male' is kept while 'Gender_female' is dropped as reference category.


,Gender_male,Embarked_Missing,Embarked_Q,Embarked_S
857,1.0,0.0,0.0,1.0
52,0.0,0.0,0.0,0.0
386,1.0,0.0,0.0,1.0
124,1.0,0.0,0.0,1.0
578,0.0,0.0,0.0,0.0


---
## 7. Summary & Best Practices Checklist

| Consideration | Recommendation |
| :--- | :--- |
| **When to Use** | Nominal categorical variables with **low-to-moderate cardinality** ($< 15$ categories). |
| **High Cardinality** | Avoid OHE if a feature has hundreds of categories; use **Frequency Encoding** (Notebook 03) or target encoding instead. |
| **Linear Models** | Use `drop='first'` to prevent perfect multicollinearity (dummy variable trap). |
| **Tree / Regularized Models** | Use `drop=None` (keep all categories); trees easily split on binary indicators without matrix inversion issues. |
| **Production Inference** | Always specify `handle_unknown='ignore'` so unexpected categories in test/production do not throw errors. |
| **Workflow Architecture** | Combine `OneHotEncoder` with `ColumnTransformer` to seamlessly integrate categorical and numerical features. |
